# Cambridge Bay -- Cross-Region Generalization: Sentinel-1 Patch Extraction

Cross-region generalization test requested by Michel: does a model
**trained only on Tuktoyaktuk** produce useful roughness predictions on a
region it has never seen? Cambridge Bay's LiDAR ground truth already
exists (provided directly, `input_data/lidar_patches_cambridge_extracted/
lidar_patches_cambridge/`, 2112 patches, verified genuinely Cambridge Bay
via CRS reprojection -- 33.4km from the townsite, correct UTM zone 13N)
-- so this notebook only needs to do the Sentinel-1 half: match PC-RTC
Sentinel-1 imagery to these *already-existing* LiDAR patches, exactly the
way `02_patch_extraction.ipynb` did for Tuktoyaktuk.

**No training happens on this data at all.** Every patch extracted here
is held out entirely for inference with an already-trained checkpoint
(`09`'s or `10`'s) in a follow-up notebook -- that's what makes this a
genuine generalization test rather than another in-region result.

**Identical logic to `02_patch_extraction.ipynb`** -- only the
region-specific config values differ (`LIDAR_DIR`, `OUT_S1_DIR`, search
date). The AOI-building, windowed VV+VH merge, and
`build_s1_products_from_corrected`/`extract_lidar_matched_s1_patches`
matching functions are copied verbatim, unmodified, so any result
difference from Tuktoyaktuk is attributable to the region itself, not a
different extraction method. The search date, `2024-04-18`, was already
present in `02`'s own `DATE_BY_REGION` dict (`{'cambridge':
dt.date(2024, 4, 18)}`), matching the `CBApr18` LiDAR survey date.

## Setup

In [1]:
import os
import json
import datetime as dt
import glob as glob_module
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

import numpy as np
import rasterio
from rasterio.warp import transform_bounds, transform_geom
from rasterio.windows import Window, from_bounds
from shapely.geometry import box, shape
from shapely.ops import unary_union

import pystac_client
import planetary_computer
from dotenv import load_dotenv

load_dotenv()
if os.environ.get('PC_SDK_SUBSCRIPTION_KEY'):
    planetary_computer.settings.set_subscription_key(os.environ['PC_SDK_SUBSCRIPTION_KEY'])

## Configuration

Only these values differ from `02_patch_extraction.ipynb`'s Tuktoyaktuk
run: `REGION`, `LIDAR_DIR` (Cambridge Bay's already-provided patches,
note the nested path from how the zip extracted), and `OUT_S1_DIR`. The
search date (`2024-04-18`) comes from the same `DATE_BY_REGION` dict `02`
already had defined.

In [2]:
REPO_DIR = Path('/cs/student/project_msc/2025/aibh/jiayiche')
INPUT_DIR = REPO_DIR / 'input_data'
REGION = 'cambridge'
LIDAR_DIR = INPUT_DIR / 'lidar_patches_cambridge_extracted' / 'lidar_patches_cambridge'
DATE_BY_REGION = {'pondinlet': dt.date(2024, 4, 26), 'cambridge': dt.date(2024, 4, 18), 'tuk': dt.date(2024, 4, 16)}
SEARCH_DAYS = 30

PATCH_SIZE = 256
S1_PATCH_SIZE = int(round(PATCH_SIZE / 10))  # 26 px @ 10m

MERGED_DIR = REPO_DIR / 'raw_data' / f'{REGION}_pc_rtc_merged'
OUT_S1_DIR = INPUT_DIR / f's1_patches_{REGION}_pcrtc'
MERGED_DIR.mkdir(parents=True, exist_ok=True)
OUT_S1_DIR.mkdir(parents=True, exist_ok=True)
print('LIDAR_DIR:', LIDAR_DIR)
print('MERGED_DIR:', MERGED_DIR)
print('OUT_S1_DIR:', OUT_S1_DIR)

LIDAR_DIR: /cs/student/project_msc/2025/aibh/jiayiche/input_data/lidar_patches_cambridge_extracted/lidar_patches_cambridge
MERGED_DIR: /cs/student/project_msc/2025/aibh/jiayiche/raw_data/cambridge_pc_rtc_merged
OUT_S1_DIR: /cs/student/project_msc/2025/aibh/jiayiche/input_data/s1_patches_cambridge_pcrtc


## 1. Build the AOI and search Planetary Computer for coverage

Identical function to `02`, just run against Cambridge Bay's LiDAR
patches instead of Tuktoyaktuk's. This automatically derives the search
area from the *actual* patch footprints, so it doesn't matter that
Cambridge Bay's patch IDs (14000-16110) or geographic extent differ from
Tuktoyaktuk's.

In [3]:
def aoi_from_lidar_patches(patches_dir, max_files=300, workers=8):
    paths = sorted(patches_dir.glob('lidar_patch_*.tif'))
    if not paths:
        raise FileNotFoundError(f'No LiDAR patches found in {patches_dir}')
    if len(paths) > max_files:
        stride = len(paths) / max_files
        paths = [paths[int(i * stride)] for i in range(max_files)]

    def read_bounds(path):
        with rasterio.open(path) as src:
            return src.crs, src.bounds

    results = []
    with ThreadPoolExecutor(max_workers=workers) as pool:
        futures = [pool.submit(read_bounds, p) for p in paths]
        for future in as_completed(futures):
            results.append(future.result())
    crs = results[0][0]
    native = unary_union([box(*bounds) for _, bounds in results])
    geojson = transform_geom(crs, 'EPSG:4326', native.__geo_interface__)
    return shape(geojson).buffer(0)

aoi = aoi_from_lidar_patches(LIDAR_DIR)
aoi_ll = aoi.convex_hull

catalog = pystac_client.Client.open(
    'https://planetarycomputer.microsoft.com/api/stac/v1',
    modifier=planetary_computer.sign_inplace,
)
start = DATE_BY_REGION[REGION] - dt.timedelta(days=SEARCH_DAYS)
end = DATE_BY_REGION[REGION] + dt.timedelta(days=SEARCH_DAYS)
search = catalog.search(collections=['sentinel-1-rtc'], intersects=aoi_ll.__geo_interface__,
                         datetime=f'{start.isoformat()}/{end.isoformat()}')
items = sorted(search.items(), key=lambda it: it.datetime)
print(f'Found {len(items)} scenes, sorted chronologically:')
for i, item in enumerate(items):
    print(f'  t{i}: {item.id} | {item.datetime}')

Found 0 scenes, sorted chronologically:


In [7]:
print('AOI valid?', aoi_ll.is_valid, '| empty?', aoi_ll.is_empty)
print('AOI bounds (lon, lat):', aoi_ll.bounds)
print('Search window:', start.isoformat(), 'to', end.isoformat())

# broaden the search to check if it's a geometry bug vs. genuine no-coverage:
# 1. try without the date filter, just the AOI
wider = catalog.search(collections=['sentinel-1-rtc'], intersects=aoi_ll.__geo_interface__)
wider_items = list(wider.items())
print(f'\nScenes for this AOI with NO date filter: {len(wider_items)}')
if wider_items:
    dates = sorted(it.datetime for it in wider_items)
    print('Earliest:', dates[0], '| Latest:', dates[-1])


AOI valid? True | empty? False
AOI bounds (lon, lat): (-105.9884493732042, 68.95816560420016, -105.67918440507977, 69.02930644230831)
Search window: 2024-03-19 to 2024-05-18

Scenes for this AOI with NO date filter: 301
Earliest: 2015-05-21 13:34:23.677559+00:00 | Latest: 2026-08-31 00:36:50.307749+00:00


In [8]:
from datetime import timezone

target = dt.datetime.combine(DATE_BY_REGION[REGION], dt.time(), tzinfo=timezone.utc)
near_target = sorted(wider_items, key=lambda it: abs((it.datetime - target).total_seconds()))[:15]
print(f'{"scene date":<30}{"days from 2024-04-18":>22}')
for it in near_target:
    days_off = (it.datetime - target).total_seconds() / 86400
    print(f'{it.datetime.isoformat():<30}{days_off:>22.1f}')


scene date                      days from 2024-04-18
2025-05-25T00:36:42.724974+00:00                 402.0
2025-05-27T00:20:24.001200+00:00                 404.0
2025-06-01T00:28:36.074235+00:00                 409.0
2025-06-13T00:28:36.647279+00:00                 421.0
2025-06-25T00:28:37.462564+00:00                 433.0
2025-06-30T00:36:44.368306+00:00                 438.0
2025-07-07T00:28:38.032819+00:00                 445.0
2025-07-12T00:36:45.442541+00:00                 450.0
2025-07-19T00:28:38.740251+00:00                 457.0
2025-07-24T00:36:46.245916+00:00                 462.0
2025-07-26T00:20:27.622554+00:00                 464.0
2025-07-31T00:28:39.551099+00:00                 469.0
2025-08-17T00:36:47.322879+00:00                 486.0
2025-08-19T00:20:28.579160+00:00                 488.0
2025-08-24T00:28:40.616232+00:00                 493.0


In [11]:
from collections import Counter
year_month_counts = Counter((it.datetime.year, it.datetime.month) for it in wider_items)
for ym in sorted(year_month_counts):
    print(ym, '->', year_month_counts[ym], 'scene(s)')


(2015, 5) -> 1 scene(s)
(2015, 11) -> 1 scene(s)
(2016, 4) -> 1 scene(s)
(2016, 8) -> 2 scene(s)
(2016, 10) -> 2 scene(s)
(2016, 11) -> 3 scene(s)
(2016, 12) -> 2 scene(s)
(2017, 1) -> 3 scene(s)
(2017, 2) -> 2 scene(s)
(2017, 3) -> 5 scene(s)
(2017, 4) -> 3 scene(s)
(2017, 5) -> 3 scene(s)
(2017, 6) -> 1 scene(s)
(2017, 8) -> 2 scene(s)
(2017, 9) -> 1 scene(s)
(2017, 10) -> 1 scene(s)
(2017, 11) -> 1 scene(s)
(2017, 12) -> 3 scene(s)
(2018, 1) -> 1 scene(s)
(2018, 2) -> 1 scene(s)
(2018, 3) -> 2 scene(s)
(2018, 4) -> 2 scene(s)
(2018, 5) -> 2 scene(s)
(2018, 6) -> 2 scene(s)
(2018, 7) -> 2 scene(s)
(2018, 8) -> 3 scene(s)
(2018, 9) -> 3 scene(s)
(2018, 10) -> 2 scene(s)
(2018, 11) -> 4 scene(s)
(2018, 12) -> 4 scene(s)
(2019, 1) -> 5 scene(s)
(2019, 2) -> 4 scene(s)
(2019, 3) -> 5 scene(s)
(2019, 4) -> 4 scene(s)
(2019, 5) -> 3 scene(s)
(2019, 6) -> 2 scene(s)
(2019, 7) -> 3 scene(s)
(2019, 8) -> 4 scene(s)
(2019, 9) -> 4 scene(s)
(2019, 10) -> 6 scene(s)
(2019, 11) -> 3 scene(s)
(201

In [12]:
pondinlet_target = dt.datetime(2024, 4, 26, tzinfo=timezone.utc)
# build a Pond Inlet AOI the same way, once you have its LiDAR patches extracted/downloaded
# (skip this check if you haven't gotten Pond Inlet's LiDAR patches onto the workstation yet)


In [14]:
import rasterio, glob
import numpy as np

files = sorted(glob.glob("/cs/student/project_msc/2025/aibh/jiayiche/input_data/lidar_patches_pondinlet_extracted/lidar_patches_pondinlet/lidar_patch_*.tif"))
print("Total patches:", len(files))
print("First ID:", files[0].split("_")[-1].replace(".tif",""))
print("Last ID:", files[-1].split("_")[-1].replace(".tif",""))

sample = files[len(files)//2]
print("\nSample file:", sample)
with rasterio.open(sample) as src:
    print("CRS:", src.crs)
    print("Resolution:", src.res)
    print("Bounds:", src.bounds)
    print("Shape:", src.shape)
    print("Band count:", src.count)
    elev = src.read(1)
    mask = src.read(2) if src.count > 1 else None
    valid = elev[mask > 0.5] if mask is not None else elev
    print("Elevation min/max/std:", valid.min(), valid.max(), valid.std())
    if mask is not None:
        print("Valid fraction:", (mask > 0.5).mean())


Total patches: 11197
First ID: 00000
Last ID: 11196

Sample file: /cs/student/project_msc/2025/aibh/jiayiche/input_data/lidar_patches_pondinlet_extracted/lidar_patches_pondinlet/lidar_patch_05598.tif
CRS: EPSG:32617
Resolution: (1.0, 1.0)
Bounds: BoundingBox(left=596579.7059493422, bottom=8076150.755047917, right=596835.7059493422, top=8076406.755047917)
Shape: (256, 256)
Band count: 2
Elevation min/max/std: -0.18266408 0.3503918 0.059406113
Valid fraction: 1.0


In [15]:
import datetime as dt
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path
from collections import Counter

import rasterio
from rasterio.warp import transform_geom
from shapely.geometry import box, shape
from shapely.ops import unary_union
import pystac_client
import planetary_computer

PONDINLET_LIDAR_DIR = Path("/cs/student/project_msc/2025/aibh/jiayiche/input_data/lidar_patches_pondinlet_extracted/lidar_patches_pondinlet")
PONDINLET_TARGET_DATE = dt.date(2024, 4, 26)

def aoi_from_lidar_patches(patches_dir, max_files=300, workers=8):
    paths = sorted(patches_dir.glob('lidar_patch_*.tif'))
    if len(paths) > max_files:
        stride = len(paths) / max_files
        paths = [paths[int(i * stride)] for i in range(max_files)]
    def read_bounds(path):
        with rasterio.open(path) as src:
            return src.crs, src.bounds
    results = []
    with ThreadPoolExecutor(max_workers=workers) as pool:
        futures = [pool.submit(read_bounds, p) for p in paths]
        for future in as_completed(futures):
            results.append(future.result())
    crs = results[0][0]
    native = unary_union([box(*bounds) for _, bounds in results])
    geojson = transform_geom(crs, 'EPSG:4326', native.__geo_interface__)
    return shape(geojson).buffer(0)

aoi_pi = aoi_from_lidar_patches(PONDINLET_LIDAR_DIR)
aoi_pi_ll = aoi_pi.convex_hull
print('Pond Inlet AOI bounds (lon, lat):', aoi_pi_ll.bounds)

catalog = pystac_client.Client.open(
    'https://planetarycomputer.microsoft.com/api/stac/v1',
    modifier=planetary_computer.sign_inplace,
)
search_pi = catalog.search(collections=['sentinel-1-rtc'], intersects=aoi_pi_ll.__geo_interface__)
items_pi = list(search_pi.items())
print(f'\nTotal scenes for Pond Inlet AOI (all time): {len(items_pi)}')

year_month_counts = Counter((it.datetime.year, it.datetime.month) for it in items_pi)
for ym in sorted(year_month_counts):
    print(ym, '->', year_month_counts[ym], 'scene(s)')


Pond Inlet AOI bounds (lon, lat): (-78.62837866403368, 72.69983676650693, -77.6792464187407, 72.85140815833994)

Total scenes for Pond Inlet AOI (all time): 98
(2015, 5) -> 1 scene(s)
(2016, 2) -> 8 scene(s)
(2016, 3) -> 6 scene(s)
(2016, 8) -> 1 scene(s)
(2016, 9) -> 1 scene(s)
(2017, 1) -> 4 scene(s)
(2017, 2) -> 10 scene(s)
(2017, 3) -> 3 scene(s)
(2018, 2) -> 7 scene(s)
(2018, 3) -> 13 scene(s)
(2019, 1) -> 1 scene(s)
(2019, 2) -> 9 scene(s)
(2019, 7) -> 1 scene(s)
(2019, 9) -> 1 scene(s)
(2020, 1) -> 3 scene(s)
(2020, 2) -> 9 scene(s)
(2021, 1) -> 1 scene(s)
(2021, 2) -> 6 scene(s)
(2021, 3) -> 1 scene(s)
(2022, 2) -> 3 scene(s)
(2022, 3) -> 2 scene(s)
(2023, 2) -> 2 scene(s)
(2023, 3) -> 2 scene(s)
(2026, 2) -> 2 scene(s)
(2026, 3) -> 1 scene(s)


In [16]:
from datetime import timezone

target_pi = dt.datetime(2024, 4, 26, tzinfo=timezone.utc)
near_target_pi = sorted(items_pi, key=lambda it: abs((it.datetime - target_pi).total_seconds()))[:10]
print(f'{"scene date":<30}{"days from 2024-04-26":>22}')
for it in near_target_pi:
    days_off = (it.datetime - target_pi).total_seconds() / 86400
    print(f'{it.datetime.isoformat():<30}{days_off:>22.1f}')


scene date                      days from 2024-04-26
2023-03-17T22:19:24.739356+00:00                -405.1
2023-03-05T22:19:24.990268+00:00                -417.1
2023-02-21T22:19:24.507218+00:00                -429.1
2023-02-09T22:19:24.836638+00:00                -441.1
2026-02-11T22:18:13.530571+00:00                 656.9
2026-02-23T22:18:13.421369+00:00                 668.9
2026-03-07T22:18:13.487702+00:00                 680.9
2022-03-22T22:19:19.476215+00:00                -765.1
2022-03-10T22:19:19.108391+00:00                -777.1
2022-02-26T22:19:19.146968+00:00                -789.1


## 2. Merge VV+VH into local 2-band GeoTIFFs, windowed to the AOI

Same windowed-HTTPS-read approach as `02` -- reads only the AOI-sized
window from each band, not the full scene.

In [4]:
merged_paths = []
merged_attrs = []

for i, item in enumerate(items):
    with rasterio.open(item.assets['vv'].href) as vv_src:
        aoi_bounds_scene_crs = transform_bounds('EPSG:4326', vv_src.crs, *aoi_ll.bounds)
        window = from_bounds(*aoi_bounds_scene_crs, transform=vv_src.transform)
        vv = vv_src.read(1, window=window)
        out_transform = rasterio.windows.transform(window, vv_src.transform)
        out_crs = vv_src.crs
    with rasterio.open(item.assets['vh'].href) as vh_src:
        vh_window = from_bounds(*transform_bounds('EPSG:4326', vh_src.crs, *aoi_ll.bounds), transform=vh_src.transform)
        vh = vh_src.read(1, window=vh_window)

    h = min(vv.shape[0], vh.shape[0])
    w = min(vv.shape[1], vh.shape[1])
    stacked = np.stack([vv[:h, :w], vh[:h, :w]]).astype(np.float32)

    out_path = MERGED_DIR / f't{i}.tif'
    meta = {'driver': 'GTiff', 'count': 2, 'height': h, 'width': w,
            'dtype': 'float32', 'crs': out_crs, 'transform': out_transform}
    with rasterio.open(out_path, 'w', **meta) as dst:
        dst.write(stacked)
    merged_paths.append(str(out_path))

    props = item.properties
    merged_attrs.append({
        'acquisition_date': item.datetime.date().isoformat(),
        'orbit_direction': 'ASCENDING' if props.get('sat:orbit_state') == 'ascending' else 'DESCENDING',
        'relative_orbit_number': props.get('sat:relative_orbit'),
    })
    print(f'Wrote t{i}.tif: shape={stacked.shape}, nodata_frac={float((stacked == -32768.0).mean()):.4f}')

attrs_json_path = MERGED_DIR / 'attrs.json'
with open(attrs_json_path, 'w') as jf:
    json.dump(merged_attrs, jf, indent=2)
print('Wrote', attrs_json_path)

Wrote /cs/student/project_msc/2025/aibh/jiayiche/raw_data/cambridge_pc_rtc_merged/attrs.json


**Check before continuing**: confirm `nodata_frac` is near 0 for every
`t{i}.tif` above. If any date shows a high nodata fraction, that scene's
footprint likely doesn't fully cover the Cambridge Bay AOI -- worth
excluding it rather than propagating a partially-empty product into the
patch matcher below.

## 3. Match against Cambridge Bay's existing LiDAR patches

Same `build_s1_products_from_corrected`/`extract_lidar_matched_s1_patches`
functions as `02`, copied verbatim -- only `LIDAR_DIR` differs.

In [5]:
def build_s1_products_from_corrected(geotiff_paths, attrs_jsons=None):
    products = []
    for i, path in enumerate(geotiff_paths):
        src = rasterio.open(path)
        attrs = attrs_jsons[i] if attrs_jsons and i < len(attrs_jsons) else None
        products.append({"src": src, "crs": src.crs, "transform": src.transform,
                          "height": src.height, "width": src.width, "attrs": attrs})
    if not products:
        raise ValueError("No Sentinel-1 products loaded -- check geotiff_paths.")
    return products


def close_products(products):
    for p in products:
        p["src"].close()


def extract_lidar_matched_s1_patches(lidar_patches_dir, sentinel1_products, s1_patch_size,
                                      out_s1_dir, pattern="lidar_patch_*.tif", max_nan_frac=0.02):
    lidar_paths = sorted(glob_module.glob(os.path.join(str(lidar_patches_dir), pattern)))
    print(f"Found {len(lidar_paths)} existing LiDAR patches to match against "
          f"{len(sentinel1_products)} Sentinel-1 product(s).")

    n_written, n_skipped, n_skipped_nan = 0, 0, 0

    for idx, lp in enumerate(lidar_paths):
        if idx % 100 == 0:
            print(f"  ...processed {idx}/{len(lidar_paths)} "
                  f"(written: {n_written}, skipped: {n_skipped}, skipped-NaN: {n_skipped_nan})")

        patch_id = os.path.splitext(os.path.basename(lp))[0].split("_")[-1]

        with rasterio.open(lp) as lsrc:
            lidar_bounds = lsrc.bounds
            lidar_crs = lsrc.crs

        s1_patches, s1_transforms = [], []
        ok = True
        has_too_much_nan = False
        for prod in sentinel1_products:
            try:
                s1_bounds = transform_bounds(lidar_crs, prod["crs"], *lidar_bounds, densify_pts=21)
                window = from_bounds(*s1_bounds, transform=prod["transform"]).round_offsets().round_lengths()
                r0, c0 = int(window.row_off), int(window.col_off)
                hh, ww = int(window.height), int(window.width)

                if (hh, ww) != (s1_patch_size, s1_patch_size):
                    ok = False; break
                if r0 < 0 or c0 < 0:
                    ok = False; break
                if r0 + s1_patch_size > prod["height"] or c0 + s1_patch_size > prod["width"]:
                    ok = False; break

                read_window = Window(c0, r0, s1_patch_size, s1_patch_size)
                patch = prod["src"].read(window=read_window)
                if patch.shape[1:] != (s1_patch_size, s1_patch_size):
                    ok = False; break

                nan_frac = float(np.mean(np.isnan(patch)))
                if nan_frac > max_nan_frac:
                    has_too_much_nan = True
                    break

                s1_patches.append(patch)
                s1_transforms.append(rasterio.windows.transform(read_window, prod["transform"]))
            except Exception:
                ok = False
                break

        if has_too_much_nan:
            n_skipped_nan += 1
            continue

        if not ok or len(s1_patches) != len(sentinel1_products):
            n_skipped += 1
            continue

        patch_dir = os.path.join(str(out_s1_dir), f"s1_patch_{patch_id}")
        os.makedirs(patch_dir, exist_ok=True)

        attrs_list = []
        for ti, (prod, patch, tr) in enumerate(zip(sentinel1_products, s1_patches, s1_transforms)):
            meta = {
                "driver": "GTiff", "count": patch.shape[0],
                "height": s1_patch_size, "width": s1_patch_size,
                "dtype": "float32", "crs": prod["crs"], "transform": tr,
            }
            with rasterio.open(os.path.join(patch_dir, f"t{ti}.tif"), "w", **meta) as dst:
                dst.write(patch.astype(np.float32))
            attrs_list.append(prod.get("attrs"))

        with open(os.path.join(patch_dir, "attrs.json"), "w") as jf:
            json.dump(attrs_list, jf, indent=2)

        n_written += 1

    print(f"Done. Matched: {n_written}, skipped (out of bounds/wrong size): {n_skipped}, "
          f"skipped (too much NaN, >{max_nan_frac:.0%}): {n_skipped_nan}")
    return n_written, n_skipped

In [6]:
products_pcrtc = build_s1_products_from_corrected(merged_paths, attrs_jsons=merged_attrs)
extract_lidar_matched_s1_patches(LIDAR_DIR, products_pcrtc, S1_PATCH_SIZE, OUT_S1_DIR)
close_products(products_pcrtc)

ValueError: No Sentinel-1 products loaded -- check geotiff_paths.

## 4. Verify the output

Same checks used throughout this project: expected fixed window size (no
CRS mismatch), and a per-band sanity check that patches aren't full of
NaN/zero placeholder values.

In [ ]:
sample_patches = sorted(glob_module.glob(os.path.join(str(OUT_S1_DIR), 's1_patch_*')))
lidar_total = len(glob_module.glob(os.path.join(str(LIDAR_DIR), 'lidar_patch_*.tif')))
print(f'Total patches written: {len(sample_patches)} (out of {lidar_total} Cambridge Bay LiDAR patches)')

if sample_patches:
    with rasterio.open(os.path.join(sample_patches[0], 't0.tif')) as src:
        print('Sample patch shape:', src.shape, f'(expect {S1_PATCH_SIZE}x{S1_PATCH_SIZE})')
    for f in sorted(glob_module.glob(os.path.join(sample_patches[0], 't*.tif'))):
        with rasterio.open(f) as src:
            arr = src.read()
            print(f'  {os.path.basename(f)}: finite_frac={np.isfinite(arr).mean():.4f}, '
                  f'nonzero_frac={(arr != 0).mean():.4f}, min/max={arr.min():.5f}/{arr.max():.5f}')

from collections import Counter
counts = Counter(len(glob_module.glob(os.path.join(p, 't*.tif'))) for p in sample_patches)
print('Distribution of timesteps per patch:', dict(sorted(counts.items())))

## Next step (separate notebook)

If the checks above look healthy (patch size correct, high
finite/nonzero fractions, no CRS-mismatch skips), the next step is a
pure-inference notebook: load `09`'s (or `10`'s) already-trained
checkpoint, run it on these Cambridge Bay patches with **no
retraining and no train/val split** (every patch is held-out test data),
and compute the same reconstruction metrics used throughout this project
for direct comparison against the Tuktoyaktuk in-region validation
numbers. `CONTEXT_K` may need reducing from `3` if fewer than 3 usable
Sentinel-1 products survive the matching step above -- check the
"Distribution of timesteps per patch" printout to confirm.